# Experiment 2 — Identical means, different shapes: paper figures

This notebook only loads `results/exp2/results.parquet` and plots — it never recomputes algorithm results (see CLAUDE.md and `docs/experiments.md` §0.4). Run `make exp2` to generate it.

Two single-row figures over the two distribution-shape problems (Problem 1: `scale_pair` σ_B = 2; Problem 2: `moment_matched_mixture` a = 0.9):
1. **`exp2_compact_row`** (main text): density panels, mean budget (τ) ± std, and mean ARI (± std, clipped to [0, 1]) for `kabc`, `kabc-sg`, `acb`.
2. **`exp2_ablation_row`** (appendix): density panels and mean budget for `kabc` vs its calibration-constant ablations (`kabc-2x`, `kabc-0.5x`).

The grey bar is the theoretical bound (the budget cap). The `ring_blob` setting is still in the results file but is not plotted.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

from kabc.environments import RingBlobEnv
from kabc.plotting import set_style, savefig, ALGO_COLORS, ALGO_LABELS

set_style()
LABELS = {**ALGO_LABELS, "kabc-2x": "KABC (2x)", "kabc-0.5x": "KABC (0.5x)"}
TITLE_FS, LABEL_FS, TICK_FS, LEGEND_FS = 10, 9, 8, 8

path = Path("../results/exp2/results.parquet")
if not path.exists():
    raise FileNotFoundError(
        f"{path} not found. Run `make exp2` -- this notebook only plots existing results, it never recomputes them."
    )
df = pd.read_parquet(path)
print(f"loaded {len(df)} rows from {path}")

MAIN_ALGOS = ["kabc", "kabc-sg", "acb"]
ABLATION_ALGOS = ["kabc", "kabc-sg", "kabc-2x", "kabc-0.5x"]
for a in MAIN_ALGOS + ABLATION_ALGOS:
    assert a in df["algorithm"].unique(), f"missing algorithm {a}"

ALL_SETTINGS = [("scale_pair", 2), ("moment_matched_mixture", 0.9), ("ring_blob", 2)]
PROBLEM_TITLES = ["Problem 1 (identical mean)", "Problem 2 (same mean, variance)", "Problem 3 (same mean, 2D shapes)"]
GROUP_LABELS = ["Problem 1", "Problem 2", "Problem 3"]
GROUP_SPACING = 1.6


def setting_df(family, swept_value):
    return df[(df["family"] == family) & (df["swept_value"] == swept_value)]


def density_panel(sf, problem_idx):
    ax = sf.subplots()
    if problem_idx == 2:
        rng = np.random.default_rng(0)
        env = RingBlobEnv(N=200, ring_radius=2, ring_noise=0.1)
        pts = np.array([env.pull(i, rng) for i in range(200)])
        ax.scatter(pts[100:, 0], pts[100:, 1], s=8, color="black", alpha=0.5, label=r"blob: $\mathcal{N}(0,I_2)$")
        ax.scatter(pts[:100, 0], pts[:100, 1], s=8, color="#2ca02c", alpha=0.6,
                   label=r"ring: radius 2, noise $\mathcal{N}(0,0.1^2 I_2)$")
        ax.set_xlabel("")
        ax.set_aspect("equal")
        ax.tick_params(labelsize=TICK_FS)
        ax.legend(fontsize=LEGEND_FS, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.08))
        sf.suptitle(PROBLEM_TITLES[2], fontsize=TITLE_FS, fontweight="bold")
        return ax
    x = np.linspace(-6, 6, 500)
    ax.plot(x, norm.pdf(x, 0, 1), color="black", linewidth=1.6, label=r"$\mathcal{N}(0,1)$")
    if problem_idx == 0:
        ax.plot(x, norm.pdf(x, 0, 2), color="#1f77b4", linestyle="--", linewidth=1.6, label=r"$\mathcal{N}(0,4)$")
    else:
        b = np.sqrt(1 - 0.9**2)
        ax.plot(x, 0.5 * norm.pdf(x, -0.9, b) + 0.5 * norm.pdf(x, 0.9, b), color="#d95f02", linestyle="--",
                linewidth=1.6, label=r"$\frac{1}{2}\mathcal{N}(-0.9,0.19)+\frac{1}{2}\mathcal{N}(0.9,0.19)$")
    ax.set_ylabel("density", fontsize=LABEL_FS)
    ax.tick_params(labelsize=TICK_FS)
    ax.legend(fontsize=LEGEND_FS, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.08))
    sf.suptitle(PROBLEM_TITLES[problem_idx], fontsize=TITLE_FS, fontweight="bold")
    return ax


def _stat(s, kind):
    mean, std = s.mean(), s.std()
    if kind == "mean_std_clip01":
        return mean, min(std, mean), min(std, 1.0 - mean)  # clip mean +/- std to [0, 1]
    return mean, std, std


def grouped_bars(ax, algos, metric, kind, with_bound, settings):
    n_bars = len(algos) + (1 if with_bound else 0)
    width = 0.8 / n_bars
    for s_idx, (family, value) in enumerate(settings):
        sub = setting_df(family, value)
        center = s_idx * GROUP_SPACING
        for a_idx, algo in enumerate(algos):
            rows = sub[sub["algorithm"] == algo]
            val, lo, hi = _stat(rows[metric], kind)
            xb = center + (a_idx - (n_bars - 1) / 2) * width
            ax.bar(xb, val, yerr=[[lo], [hi]], width=width * 0.92, capsize=2, color=ALGO_COLORS[algo], alpha=0.88)
            if metric == "ari" and val < 0.005:
                ax.annotate("0", (xb, 0), xytext=(0, 2), textcoords="offset points", ha="center", va="bottom",
                            fontsize=TICK_FS)
        if with_bound:
            bound = sub[sub["algorithm"].isin(algos)]["budget_cap"].mean()
            xb = center + (len(algos) - (n_bars - 1) / 2) * width
            ax.bar(xb, bound, width=width * 0.92, color="grey", alpha=0.6)
    ax.set_xticks([i * GROUP_SPACING for i in range(len(settings))])
    ax.set_xticklabels(GROUP_LABELS[:len(settings)], fontsize=LABEL_FS)
    ax.tick_params(axis="y", labelsize=TICK_FS)


def shared_legend(fig, algos):
    handles = [plt.Rectangle((0, 0), 1, 1, color=ALGO_COLORS[a], alpha=0.88, label=LABELS[a]) for a in algos]
    handles.append(plt.Rectangle((0, 0), 1, 1, color="grey", alpha=0.6, label="theoretical bound"))
    fig.legend(handles=handles, loc="outside lower center", ncol=len(handles), frameon=False, fontsize=LEGEND_FS)


df.head()

## Main text: density panels, budget (τ) and ARI

In [ ]:
fig = plt.figure(figsize=(14, 3.4), layout="constrained")
sf1, sf2, sf3 = fig.subfigures(1, 3, width_ratios=[1, 1, 2.2], wspace=0.02)
density_panel(sf1, 0)
density_panel(sf2, 1)

ax_tau, ax_ari = sf3.subplots(1, 2)
grouped_bars(ax_tau, MAIN_ALGOS, "tau", "mean_std", with_bound=True, settings=ALL_SETTINGS[:2])
ax_tau.set_yscale("log")
ax_tau.set_ylabel(r"mean budget ($\tau$) ± std", fontsize=LABEL_FS)
grouped_bars(ax_ari, MAIN_ALGOS, "ari", "mean_std_clip01", with_bound=False, settings=ALL_SETTINGS[:2])
ax_ari.set_ylim(0, 1.05)
ax_ari.set_ylabel("mean ARI (± std)", fontsize=LABEL_FS)
sf3.suptitle("Comparison of algorithms", fontsize=TITLE_FS, fontweight="bold")

shared_legend(fig, MAIN_ALGOS)
savefig(fig, "exp2_compact_row", fig_dir="../results/figures")
plt.show()

## Appendix: calibration-constant ablation

In [ ]:
fig = plt.figure(figsize=(14, 3.4), layout="constrained")
sf1, sf2, sf3, sf4 = fig.subfigures(1, 4, width_ratios=[1, 1, 1, 1.9], wspace=0.02)
density_panel(sf1, 0)
density_panel(sf2, 1)
density_panel(sf3, 2)

ax = sf4.subplots()
grouped_bars(ax, ABLATION_ALGOS, "tau", "mean_std", with_bound=True, settings=ALL_SETTINGS)
ax.set_yscale("log")
ax.set_ylabel(r"mean budget ($\tau$) ± std", fontsize=LABEL_FS)
sf4.suptitle("Calibration-constant ablation", fontsize=TITLE_FS, fontweight="bold")

shared_legend(fig, ABLATION_ALGOS)
savefig(fig, "exp2_ablation_row", fig_dir="../results/figures")
plt.show()